# LSTM 시계열 매출 예측 모델

일별 총 매출을 시계열로 학습해 향후 매출을 예측합니다.

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

DATA_DIR   = "/kaggle/input/gyeonggi-card-data"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(f"{OUTPUT_DIR}/model", exist_ok=True)

SEQ_LEN    = 30   # 과거 30일로 다음 날 예측
EPOCHS     = 50
BATCH_SIZE = 64
LR         = 0.001

files = sorted(glob.glob(f"{DATA_DIR}/tbsh_gyeonggi_day_*.csv"))
print(f"파일 수: {len(files)}개")

In [ ]:
# 1) 전체 파일에서 날짜별 총 매출 집계
dfs = []
for f in files:
    df = pd.read_csv(f, encoding="utf-8-sig", usecols=["ta_ymd", "amt"],
                     dtype={"amt": "int64"})
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["ta_ymd"].astype(str), format="%Y%m%d")
daily = df_all.groupby("date")["amt"].sum().reset_index().sort_values("date")
daily = daily.dropna()
print(f"날짜 범위: {daily['date'].min().date()} ~ {daily['date'].max().date()}")
print(f"총 {len(daily)}일")

In [ ]:
# 2) 시퀀스 생성
values = daily["amt"].values.astype("float32").reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, SEQ_LEN)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)
print(f"학습: {X_train.shape}  검증: {X_test.shape}")

In [ ]:
# 3) LSTM 모델 정의
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = LSTMModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Device: {device}")

In [ ]:
# 4) 학습
from torch.utils.data import DataLoader, TensorDataset

train_ds = TensorDataset(X_train.to(device), y_train.to(device))
loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

t0 = time.time()
for epoch in range(1, EPOCHS+1):
    model.train()
    losses = []
    for xb, yb in loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_test.to(device)), y_test.to(device)).item()
        print(f"Epoch {epoch:3d}  train_loss={np.mean(losses):.6f}  val_loss={val_loss:.6f}  ({time.time()-t0:.0f}s)")

In [ ]:
# 5) 저장 (CPU로 이동 후 저장)
model.cpu()
joblib.dump({
    "model":   model,
    "scaler":  scaler,
    "seq_len": SEQ_LEN,
    "history": daily["amt"].values.astype("float32"),
    "dates":   daily["date"].dt.strftime("%Y-%m-%d").tolist(),
}, f"{OUTPUT_DIR}/model/lstm_model.pkl")
print("저장 완료: model/lstm_model.pkl")